# Sentiment Engine

Combines all news sources into one unified daily sentiment signal per ticker.

## Pipeline
```
HN + Reddit + GDELT + StockTwits + EDGAR
         ↓
  Fuzzy company matching (all 3,800+ tickers)
         ↓
  VADER scoring per source
         ↓
  Adaptive rolling windows (sized by momentum + volume surge)
         ↓
  Combined weighted sentiment signal
         ↓
  GitHub: sentiment_outputs/daily_sentiment.csv
```

## Source weights (data-driven, configurable)
| Source | Default weight | Best for |
|--------|---------------|----------|
| HN | 1.0 | Tech companies, expert signal |
| Reddit/WSB | 0.8 | Retail momentum, meme stocks |
| Reddit/investing | 1.0 | Broader market, more signal |
| GDELT | 0.9 | Macro events, mainstream news |
| StockTwits | 1.1 | Direct ticker sentiment, highest relevance |
| EDGAR 8-K | 1.5 | Hard facts, material events |

---
## 0. Install

In [1]:
# !pip install vaderSentiment rapidfuzz pandas requests python-dotenv

---
## 1. Configuration

In [2]:
import os, io, re, requests, time, warnings
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
load_dotenv()

GITHUB_REPO    = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
GITHUB_TOKEN   = os.environ.get('GITHUB_TOKEN', None)

# Folder prefixes
HN_PREFIX        = 'hn_data'
REDDIT_PREFIX    = 'reddit_data'
GDELT_PREFIX     = 'gdelt_data'
STOCKTWITS_PREFIX= 'stocktwits_data'
EDGAR_PREFIX     = 'edgar_data'
STOCKS_PREFIX    = 'stocks'
OUTPUT_PREFIX    = 'sentiment_outputs'

# Analysis window
ANALYSIS_START = '2015-01-01'
ANALYSIS_END   = datetime.now(timezone.utc).strftime('%Y-%m-%d')

# Source weights
SOURCE_WEIGHTS = {
    'hn'           : 1.0,
    'reddit_wsb'   : 0.8,
    'reddit_stocks': 1.0,
    'reddit_invest': 1.0,
    'reddit_tech'  : 0.7,
    'reddit_sa'    : 1.1,
    'gdelt'        : 0.9,
    'stocktwits'   : 1.1,
    'edgar_8k'     : 1.5,
}

# Adaptive window settings
BASE_WINDOW       = 5     # baseline rolling window days
MAX_WINDOW        = 21    # maximum window when momentum is low
MIN_WINDOW        = 2     # minimum window when momentum is high
VOLUME_SURGE_MULT = 0.5   # shrink window by this fraction on volume surge

# Matching thresholds
FUZZY_THRESHOLD   = 90    # raised from 85 — reduces false name matches
SENTIMENT_THRESHOLD = 0.05

print('Configuration loaded')
print(f'  Repo           : {GITHUB_REPO}')
print(f'  Token          : {"set" if GITHUB_TOKEN else "NOT SET"}')
print(f'  Analysis window: {ANALYSIS_START} to {ANALYSIS_END}')
print(f'  Source weights : {SOURCE_WEIGHTS}')

Configuration loaded
  Repo           : annhmartin/dataviz-historical-stocks-AnnetteMartin
  Token          : set
  Analysis window: 2015-01-01 to 2026-07-16
  Source weights : {'hn': 1.0, 'reddit_wsb': 0.8, 'reddit_stocks': 1.0, 'reddit_invest': 1.0, 'reddit_tech': 0.7, 'reddit_sa': 1.1, 'gdelt': 0.9, 'stocktwits': 1.1, 'edgar_8k': 1.5}


---
## 2. GitHub helpers

In [3]:
GITHUB_API = 'https://api.github.com'

def _gh_headers(token):
    return {'Authorization': f'Bearer {token}',
            'Accept': 'application/vnd.github+json',
            'X-GitHub-Api-Version': '2022-11-28'}

def load_csv(path, token=None):
    url = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{path}'
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.status_code == 404: raise FileNotFoundError(path)
    resp.raise_for_status()
    content = resp.text.strip()
    if not content: return pd.DataFrame()
    return pd.read_csv(io.StringIO(content), low_memory=False)

def push_csv(df, path, token, msg=None):
    if msg is None:
        msg = f'Update {path} - {len(df):,} rows [{datetime.now(timezone.utc).strftime("%Y-%m-%d")}]'
    buf = io.StringIO()
    df.to_csv(buf, index=False)
    encoded = __import__('base64').b64encode(buf.getvalue().encode()).decode()
    url = f'{GITHUB_API}/repos/{GITHUB_REPO}/contents/{path}'
    headers = _gh_headers(token)
    check = requests.get(url, headers=headers, timeout=15)
    sha = check.json().get('sha') if check.status_code == 200 else None
    payload = {'message': msg, 'content': encoded}
    if sha: payload['sha'] = sha
    for attempt in range(3):
        resp = requests.put(url, headers=headers, json=payload, timeout=120)
        if resp.status_code in (200, 201):
            print(f'  Saved {path} ({len(df):,} rows)')
            return True
        if resp.status_code == 409 and attempt < 2:
            time.sleep(3)
            check = requests.get(url, headers=headers, timeout=15)
            sha = check.json().get('sha') if check.status_code == 200 else None
            if sha: payload['sha'] = sha
        else:
            print(f'  FAILED {path}: {resp.status_code}')
            return False

print('GitHub helpers loaded')

GitHub helpers loaded


---
## 3. Build Expanded Ticker Universe

Uses SEC EDGAR company database + fuzzy matching to expand from 24 to 500+ tickers.
Matches any text mentioning a company name (including misspellings) to its ticker.

Three matching layers:
1. **Exact ticker symbol** (e.g. NVDA, $NVDA)
2. **Official company name** from SEC EDGAR (exact + fuzzy)
3. **Common aliases and product names** from an extended keywords list

In [4]:
from rapidfuzz import fuzz, process

print('Building expanded ticker universe from SEC EDGAR ...')

# Load SEC company tickers (all public companies)
sec_url = 'https://www.sec.gov/files/company_tickers.json'
resp = requests.get(sec_url, headers={'User-Agent': 'TechPulse/1.0 (research@example.com)'}, timeout=15)
sec_data = resp.json()

# Build full company name → ticker mapping
sec_map = {}
for entry in sec_data.values():
    ticker = entry.get('ticker','').upper()
    title  = entry.get('title','').strip()
    if ticker and title:
        sec_map[ticker] = title

print(f'SEC EDGAR: {len(sec_map):,} public companies')

# Load your stock price index to get tickers with actual price data
try:
    df_stk_idx = load_csv(f'{STOCKS_PREFIX}/index.csv', GITHUB_TOKEN)
    priced_tickers = set(df_stk_idx['ticker'].dropna().str.upper().tolist())
    print(f'Tickers with price data: {len(priced_tickers):,}')
except FileNotFoundError:
    priced_tickers = set(sec_map.keys())
    print('No price index found — using all SEC tickers')

# Build TICKER_UNIVERSE: only tickers we have prices for
TICKER_UNIVERSE = {
    ticker: sec_map[ticker]
    for ticker in priced_tickers
    if ticker in sec_map
}
print(f'Matched universe: {len(TICKER_UNIVERSE):,} tickers with both price data and SEC name')

# Build fuzzy matching index
# key = normalized company name, value = ticker
def normalize(text):
    text = text.lower()
    # Remove common suffixes that add noise
    for suffix in [' inc', ' inc.', ' corp', ' corp.', ' corporation', ' ltd',
                   ' llc', ' co.', ' co', ' plc', ' group', ' holdings',
                   ' technologies', ' technology', ' systems', ' solutions',
                   ' international', ' global', ' services']:
        text = text.replace(suffix, '')
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

name_to_ticker = {normalize(name): ticker
                  for ticker, name in TICKER_UNIVERSE.items()}

# Also add ticker symbols directly
ticker_set = set(TICKER_UNIVERSE.keys())

print(f'Fuzzy match index built: {len(name_to_ticker):,} normalized names')
print('\nSample entries:')
for i, (k, v) in enumerate(list(name_to_ticker.items())[:10]):
    print(f'  {k:40s} -> {v}')

# Manual overrides for companies with unusual SEC names or popular aliases
# These bypass fuzzy matching for high-confidence known aliases
MANUAL_ALIASES = {
    # Pharma / biotech
    'novo nordisk'   : 'NVO',  'ozempic': 'NVO', 'wegovy': 'NVO', 'semaglutide': 'NVO',
    'eli lilly'      : 'LLY',  'mounjaro': 'LLY', 'tirzepatide': 'LLY',
    'pfizer'         : 'PFE',  'moderna': 'MRNA', 'johnson johnson': 'JNJ',
    'abbvie'         : 'ABBV', 'bristol myers': 'BMY', 'merck': 'MRK',
    # Tech
    'alphabet'       : 'GOOGL', 'deepmind': 'GOOGL', 'waymo': 'GOOGL',
    'meta'           : 'META',  'instagram': 'META', 'whatsapp': 'META', 'oculus': 'META',
    'aws'            : 'AMZN',  'amazon web services': 'AMZN', 'prime video': 'AMZN',
    'azure'          : 'MSFT',  'github': 'MSFT', 'linkedin': 'MSFT', 'openai': 'MSFT',
    'iphone'         : 'AAPL',  'macos': 'AAPL', 'app store': 'AAPL', 'airpods': 'AAPL',
    'chatgpt'        : 'MSFT',  'gpt-4': 'MSFT', 'gpt4': 'MSFT',
    'gemini'         : 'GOOGL', 'bard': 'GOOGL',
    'llama'          : 'META',  'pytorch': 'META',
    'cuda'           : 'NVDA',  'h100': 'NVDA', 'a100': 'NVDA', 'blackwell': 'NVDA',
    'snapdragon'     : 'QCOM',  'arm chip': 'QCOM',
    'tsmc'           : 'TSM',   'taiwan semiconductor': 'TSM',
    'ryzen'          : 'AMD',   'epyc': 'AMD', 'radeon': 'AMD',
    'xeon'           : 'INTC',  'intel core': 'INTC',
    # Finance / crypto
    'bitcoin'        : 'COIN',  'ethereum': 'COIN', 'crypto': 'COIN',
    'venmo'          : 'PYPL',  'paypal': 'PYPL',
    # Biotech / pharma portfolio
    'jakafi'         : 'INCY',  'ruxolitinib': 'INCY',
    'iqos'           : 'PM',    'heated tobacco': 'PM', 'zyn': 'PM',
    'kinross'        : 'KGC',
    'wheaton'        : 'WPM',   'silver streaming': 'WPM',
    # Cloud / SaaS
    'snowflake'      : 'SNOW',  'snowpark': 'SNOW',
    'datadog'        : 'DDOG',  'crowdstrike': 'CRWD', 'falcon': 'CRWD',
    'palantir'       : 'PLTR',  'gotham': 'PLTR', 'foundry': 'PLTR',
    'palo alto'      : 'PANW',  'prisma cloud': 'PANW',
    'servicenow'     : 'NOW',   'mongodb': 'MDB', 'atlas database': 'MDB',
    'okta'           : 'OKTA',  'salesforce': 'CRM', 'slack': 'CRM',
    'tesla'          : 'TSLA',  'autopilot': 'TSLA', 'gigafactory': 'TSLA',
    'netflix'        : 'NFLX',  'spotify': 'SPOT', 'coinbase': 'COIN',
}
# Add manual aliases to name_to_ticker (they take priority)
for alias, ticker in MANUAL_ALIASES.items():
    name_to_ticker[alias] = ticker
print(f'Manual aliases added: {len(MANUAL_ALIASES)}')
print(f'Total name-to-ticker entries: {len(name_to_ticker):,}')

Building expanded ticker universe from SEC EDGAR ...
SEC EDGAR: 10,426 public companies
Tickers with price data: 3,850
Matched universe: 2,689 tickers with both price data and SEC name
Fuzzy match index built: 2,534 normalized names

Sample entries:
  americanastal insurance                  -> ACIC
  beasley broadcast                        -> BBGI
  comstock holdingmpanies                  -> CHCI
  mustang bio                              -> MBIO
  wafd                                     -> WAFD
  versabank                                -> VBNK
  republic power                           -> RPGL
  aspac iii acquisition                    -> ASPC
  gaxos ai                                 -> GXAI
  cipher digital                           -> CIFR
Manual aliases added: 86
Total name-to-ticker entries: 2,609


In [5]:
# Common English words that are also valid ticker symbols — exclude from matching
WORD_BLOCKLIST = {
    'A','I','ON','IN','AT','IS','IT','BE','AS','AN','OR','IF','NO','TO','DO',
    'GO','SO','WE','HE','ME','MY','BY','UP','US','AM','PM','VS','CAN','HAS',
    'HAD','WAS','ARE','FOR','THE','AND','BUT','NOT','ANY','ALL','NEW','NOW',
    'ONE','TWO','WAY','DAY','MAN','OUT','GET','GOT','PUT','SET','LET','RUN',
    'HOW','WHY','WHO','WHAT','WHEN','WILL','WELL','GOOD','REAL','OPEN','HELP',
    'WORK','LIFE','NEXT','LAST','LONG','HIGH','JUST','OVER','BACK','ALSO',
    'BOTH','EVEN','EACH','MUCH','SUCH','THAN','THEN','SOME','HERE','THEY',
    'FROM','WITH','HAVE','THIS','THAT','BEEN','DOES','SAID','MAKE','MADE',
    'TAKE','TOOK','CAME','COME','CALL','LOOK','NEED','WANT','KNOW','SHOW',
    'LOVE','LIVE','MOVE','HOLD','SELL','SOLD','FIND','GIVE','GAVE','TELL',
    'PLAY','STAY','KEEP','GROW','FALL','FEEL','SEEM','TURN','COST','ONLY',
    'MANY','MORE','MOST','VERY','FULL','FREE','SAFE','TRUE','BEST','NEXT',
    'TOP','BIG','APP','NET','WEB','API','CEO','CFO','COO','IPO','ETF','GDP',
    'ANY','END','OLD','OWN','OFF','ODD','ADD','ACT','AIM','AGE','AIR','ARM',
    'ART','ASK','BAD','BAR','BASE','BEAT','BILL','BIT','BOX','BUY','CASH',
    'CHAT','CHIP','CUT','DATA','DEAL','DEBT','DEEP','DIPS','DROP','DRUG',
    'EARN','EASY','EDGE','ELSE','FAST','FILE','FILL','FIRM','FLAT','FLOW',
    'FUND','GAIN','GAME','GOLD','GOOG','GRID','GROW','HACK','HARD','HEAD',
    'HEAT','HITS','HOME','HOST','HUGE','IDEA','INFO','INIT','INTO','JOBS',
    'JOIN','JUMP','KEYS','KIND','KNOW','LAND','LEAD','LEAN','LEFT','LESS',
    'LIKE','LINE','LINK','LIST','LOAD','LOCK','LOSS','LOST','LOTS','LOW',
    'MAIN','MARK','MASS','MEAN','MEET','MIND','MINE','MISS','MODE','MONEY',
    'MORE','MOVE','NAME','NEWS','NICE','NOPE','NORM','NOTE','ONCE','ONLY',
    'OPEN','PAID','PAIN','PART','PASS','PATH','PEAK','PICK','PLAN','PLUS',
    'POOL','POOR','PORT','PUSH','RATE','READ','RISK','ROAD','ROLE','ROOM',
    'ROOT','RULE','RUNS','RUSH','SAFE','SAME','SAVE','SAYS','SEES','SEND',
    'SENT','SIGN','SIZE','SKIP','SLOW','SNAP','SORT','SPIN','SPOT','STOP',
    'SUMS','SURE','SWAP','TALK','TASK','TEAM','TECH','TEST','TEXT','THAN',
    'THEM','THEN','THEY','THIN','TIES','TIME','TIPS','TOLD','TOOL','TOPS',
    'TOWN','TREE','TRIM','TRIP','TRUE','TUNE','TYPE','UNIT','UPON','USED',
    'USES','VIEW','VOTE','WAIT','WALK','WAVE','WAYS','WEEK','WHAT','WHEN',
    'WHOM','WIDE','WINS','WISE','WISH','WITH','WORD','WORE','WORN','WRAP',
    'YEAR','ZERO','ZONE','STHO','PPLI','RDIB','NWSA',
}

def find_tickers_in_text(text, threshold=FUZZY_THRESHOLD):
    """
    Find all ticker mentions in a text string using three layers:
    1. Exact ticker symbol match ($NVDA or standalone NVDA) — min 3 chars, not a common word
    2. Exact company name match after normalization
    3. Fuzzy company name match above threshold
    Returns list of (ticker, confidence, match_type) tuples.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    found = {}
    text_clean = re.sub(r'[^a-zA-Z0-9$\s]', ' ', text)
    words = set(text_clean.upper().split())

    # Layer 1: Exact ticker match — min 3 chars, not a common English word
    for word in words:
        clean_word = word.lstrip('$')
        if (clean_word in ticker_set
                and len(clean_word) >= 3
                and clean_word not in WORD_BLOCKLIST):
            found[clean_word] = (100, 'ticker_exact')

    # Layer 1b: Manual alias exact match (product names, popular names)
    text_lower = text.lower()
    for alias, ticker in MANUAL_ALIASES.items():
        if alias in text_lower and ticker not in found:
            found[ticker] = (100, 'alias_exact')

    # Layer 2 + 3: Company name matching
    text_norm = normalize(text)
    if len(text_norm) > 3:
        # Try sliding window of 1-5 words for company name matches
        words_norm = text_norm.split()
        for window in range(1, min(6, len(words_norm)+1)):
            for i in range(len(words_norm) - window + 1):
                phrase = ' '.join(words_norm[i:i+window])
                if len(phrase) < 3: continue

                # Exact match
                if phrase in name_to_ticker:
                    ticker = name_to_ticker[phrase]
                    if ticker not in found:
                        found[ticker] = (100, 'name_exact')
                    continue

                # Fuzzy match (only for phrases >= 5 chars to avoid false positives)
                if len(phrase) >= 5:
                    result = process.extractOne(
                        phrase, name_to_ticker.keys(),
                        scorer=fuzz.token_sort_ratio,
                        score_cutoff=threshold
                    )
                    if result:
                        match_name, score, _ = result
                        ticker = name_to_ticker[match_name]
                        if ticker not in found or score > found[ticker][0]:
                            found[ticker] = (score, 'name_fuzzy')

    return [(ticker, conf, mtype) for ticker, (conf, mtype) in found.items()]

# Test the matcher
test_cases = [
    'Nvidia releases new H100 GPU for AI training',
    'NVDA stock hits all time high',
    'Microsft Azure cloud growing fast',  # intentional misspelling
    'Apple Iphone 16 sales disappointing',
    'bought some $TSLA calls this morning',
    'CrowdStrike outage affected millions of computers',
    'Novo Nordisk Ozempic weight loss drug demand',
]
print('Testing fuzzy matcher:')
for text in test_cases:
    matches = find_tickers_in_text(text)
    print(f'  "{text[:60]}"')
    for ticker, conf, mtype in matches:
        print(f'    -> {ticker} (score={conf}, type={mtype})')

Testing fuzzy matcher:
  "Nvidia releases new H100 GPU for AI training"
    -> NVDA (score=100, type=alias_exact)
  "NVDA stock hits all time high"
    -> NVDA (score=100, type=ticker_exact)
  "Microsft Azure cloud growing fast"
    -> MSFT (score=100, type=alias_exact)
  "Apple Iphone 16 sales disappointing"
    -> AAPL (score=100, type=alias_exact)
  "bought some $TSLA calls this morning"
    -> TSLA (score=100, type=ticker_exact)
  "CrowdStrike outage affected millions of computers"
    -> CRWD (score=100, type=alias_exact)
  "Novo Nordisk Ozempic weight loss drug demand"
    -> NVO (score=100, type=alias_exact)


---
## 4. Load & Score All News Sources

Loads each source, runs VADER sentiment, matches to tickers using fuzzy matching.
Each story gets: ticker, date, sentiment score, source, confidence.

In [6]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def score_text(text):
    if not isinstance(text, str) or not text.strip(): return 0.0
    return analyzer.polarity_scores(str(text))['compound']

def process_source(df, text_col, date_col, source_name, weight,
                   extra_cols=None, batch_size=1000):
    """
    Score a source DataFrame and match to tickers.
    Returns long-format DataFrame with one row per story-ticker match.
    """
    if df.empty: return pd.DataFrame()

    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col, text_col])
    df = df[(df[date_col] >= ANALYSIS_START) & (df[date_col] <= ANALYSIS_END)]
    if df.empty: return pd.DataFrame()

    print(f'  Scoring {source_name}: {len(df):,} items ...')
    df['sentiment'] = df[text_col].fillna('').apply(score_text)

    rows = []
    for i, (_, row) in enumerate(df.iterrows()):
        if i % batch_size == 0 and i > 0:
            print(f'    {i:,}/{len(df):,} processed ...', end='\r')
        text    = str(row[text_col])
        matches = find_tickers_in_text(text)
        for ticker, confidence, match_type in matches:
            entry = {
                'ticker'     : ticker,
                'date'       : row[date_col].date(),
                'sentiment'  : row['sentiment'],
                'source'     : source_name,
                'weight'     : weight,
                'confidence' : confidence,
                'match_type' : match_type,
                'text_snippet': text[:200],
            }
            if extra_cols:
                for col in extra_cols:
                    entry[col] = row.get(col)
            rows.append(entry)

    result = pd.DataFrame(rows)
    if not result.empty:
        result['date'] = pd.to_datetime(result['date'])
    print(f'  {source_name}: {len(result):,} ticker matches from {len(df):,} items')
    return result

all_scored = []
start_year = int(ANALYSIS_START[:4])
end_year   = int(ANALYSIS_END[:4])

In [7]:
# Load and score HN
print('\n=== Loading HN ===')
hn_frames = []
for year in range(start_year, end_year + 1):
    try:
        df_y = load_csv(f'{HN_PREFIX}/hn_{year}.csv', GITHUB_TOKEN)
        hn_frames.append(df_y)
    except FileNotFoundError: pass

if hn_frames:
    df_hn = pd.concat(hn_frames, ignore_index=True).drop_duplicates(subset='id')
    scored_hn = process_source(df_hn, 'title', 'date', 'hn',
                               SOURCE_WEIGHTS['hn'], extra_cols=['points'])
    all_scored.append(scored_hn)
    print(f'HN: {len(df_hn):,} stories -> {len(scored_hn):,} matches')

# Load and score Reddit (split by subreddit for weighting)
print('\n=== Loading Reddit ===')
reddit_frames = []
for year in range(start_year, end_year + 1):
    try:
        df_y = load_csv(f'{REDDIT_PREFIX}/reddit_{year}.csv', GITHUB_TOKEN)
        reddit_frames.append(df_y)
    except FileNotFoundError: pass

if reddit_frames:
    df_reddit = pd.concat(reddit_frames, ignore_index=True).drop_duplicates(subset='id')
    # Apply per-subreddit weights
    sub_weight_map = {
        'wallstreetbets': SOURCE_WEIGHTS['reddit_wsb'],
        'stocks'        : SOURCE_WEIGHTS['reddit_stocks'],
        'investing'     : SOURCE_WEIGHTS['reddit_invest'],
        'technology'    : SOURCE_WEIGHTS['reddit_tech'],
        'SecurityAnalysis': SOURCE_WEIGHTS['reddit_sa'],
    }
    for sub, weight in sub_weight_map.items():
        df_sub = df_reddit[df_reddit['subreddit']==sub]
        if df_sub.empty: continue
        scored = process_source(df_sub, 'title', 'date', f'reddit_{sub}',
                                weight, extra_cols=['score'])
        all_scored.append(scored)
    print(f'Reddit: {len(df_reddit):,} posts processed')


=== Loading HN ===
  Scoring hn: 664,598 items ...
  hn: 174,203 ticker matches from 664,598 items
HN: 664,599 stories -> 174,203 matches

=== Loading Reddit ===
  Scoring reddit_wallstreetbets: 131,592 items ...
  reddit_wallstreetbets: 36,711 ticker matches from 131,592 items
  Scoring reddit_stocks: 105,172 items ...
  reddit_stocks: 30,692 ticker matches from 105,172 items
  Scoring reddit_investing: 125,369 items ...
  reddit_investing: 27,414 ticker matches from 125,369 items
  Scoring reddit_technology: 138,825 items ...
  reddit_technology: 54,530 ticker matches from 138,825 items
  Scoring reddit_SecurityAnalysis: 20,034 items ...
  reddit_SecurityAnalysis: 4,526 ticker matches from 20,034 items
Reddit: 520,992 posts processed


In [8]:
# Load and score GDELT
print('\n=== Loading GDELT ===')
gdelt_frames = []
for ticker in list(TICKER_UNIVERSE.keys()):  # load all tickers with GDELT data
    for year in range(max(start_year, 2013), end_year + 1):
        try:
            df_g = load_csv(f'{GDELT_PREFIX}/gdelt_{ticker}_{year}.csv', GITHUB_TOKEN)
            df_g['source_ticker'] = ticker  # GDELT already matched to ticker
            gdelt_frames.append(df_g)
        except FileNotFoundError: pass

if gdelt_frames:
    df_gdelt = pd.concat(gdelt_frames, ignore_index=True)
    df_gdelt['date'] = pd.to_datetime(df_gdelt['date'], errors='coerce')
    df_gdelt = df_gdelt.dropna(subset=['date'])
    df_gdelt = df_gdelt[(df_gdelt['date'] >= ANALYSIS_START) & (df_gdelt['date'] <= ANALYSIS_END)]
    # GDELT is pre-matched to tickers — use tone directly, normalize to -1/+1
    df_gdelt['sentiment'] = (df_gdelt['tone'] / 10).clip(-1, 1)
    gdelt_long = df_gdelt[['source_ticker','date','sentiment','title']].copy()
    gdelt_long.columns = ['ticker','date','sentiment','text_snippet']
    gdelt_long['source']     = 'gdelt'
    gdelt_long['weight']     = SOURCE_WEIGHTS['gdelt']
    gdelt_long['confidence'] = 100
    gdelt_long['match_type'] = 'pre_matched'
    all_scored.append(gdelt_long)
    print(f'GDELT: {len(gdelt_long):,} articles matched')

# Load StockTwits
print('\n=== Loading StockTwits ===')
try:
    df_st_idx = load_csv(f'{STOCKTWITS_PREFIX}/stocktwits_index.csv', GITHUB_TOKEN)
    st_frames = []
    for ticker in df_st_idx['ticker'].tolist():
        try:
            df_st = load_csv(f'{STOCKTWITS_PREFIX}/stocktwits_{ticker}.csv', GITHUB_TOKEN)
            st_frames.append(df_st)
        except FileNotFoundError: pass
    if st_frames:
        df_stocktwits = pd.concat(st_frames, ignore_index=True)
        # StockTwits are pre-tagged with ticker — score body text
        df_stocktwits['sentiment'] = df_stocktwits['body'].fillna('').apply(score_text)
        df_stocktwits['date'] = pd.to_datetime(df_stocktwits['created_at'], errors='coerce')
        st_long = df_stocktwits[['ticker','date','sentiment']].copy()
        st_long['source']       = 'stocktwits'
        st_long['weight']       = SOURCE_WEIGHTS['stocktwits']
        st_long['confidence']   = 100
        st_long['match_type']   = 'ticker_tagged'
        st_long['text_snippet'] = df_stocktwits['body'].fillna('').str[:200]
        all_scored.append(st_long)
        print(f'StockTwits: {len(st_long):,} messages')
except FileNotFoundError:
    print('StockTwits: no data yet — run stocktwits_collector.ipynb first')

# EDGAR 8-K boost events
print('\n=== Loading EDGAR ===')
edgar_frames = []
for ticker in list(TICKER_UNIVERSE.keys()):
    try:
        df_e = load_csv(f'{EDGAR_PREFIX}/edgar_{ticker}.csv', GITHUB_TOKEN)
        df_e['ticker'] = ticker
        edgar_frames.append(df_e[df_e['form']=='8-K'][['ticker','date','form']])
    except FileNotFoundError: pass

if edgar_frames:
    df_edgar = pd.concat(edgar_frames, ignore_index=True)
    df_edgar['date'] = pd.to_datetime(df_edgar['date'], errors='coerce')
    df_edgar = df_edgar.dropna(subset=['date'])
    # 8-K events get a fixed positive signal boost weight — no sentiment score
    # They are event markers, not sentiment
    df_edgar['sentiment']   = 0.0   # neutral — just marks the event happened
    df_edgar['source']      = 'edgar_8k'
    df_edgar['weight']      = SOURCE_WEIGHTS['edgar_8k']
    df_edgar['confidence']  = 100
    df_edgar['match_type']  = 'sec_filing'
    df_edgar['text_snippet']= '8-K Material Event Filing'
    all_scored.append(df_edgar[['ticker','date','sentiment','source','weight','confidence','match_type','text_snippet']])
    print(f'EDGAR: {len(df_edgar):,} 8-K filing events')

# Combine all
df_all_raw = pd.concat(all_scored, ignore_index=True)
df_all_raw['date'] = pd.to_datetime(df_all_raw['date'])
df_all_raw['weighted_sent'] = df_all_raw['sentiment'] * df_all_raw['weight']
print(f'\nTotal matched items: {len(df_all_raw):,}')
print(f'Unique tickers matched: {df_all_raw["ticker"].nunique():,}')
print(f'Date range: {df_all_raw["date"].min().date()} to {df_all_raw["date"].max().date()}')
print('\nItems per source:')
print(df_all_raw.groupby('source')['ticker'].count().sort_values(ascending=False).to_string())


=== Loading GDELT ===
GDELT: 241,536 articles matched

=== Loading StockTwits ===
StockTwits: 2,312 messages

=== Loading EDGAR ===
EDGAR: 3,730 8-K filing events

Total matched items: 575,654
Unique tickers matched: 2,006
Date range: 1994-01-27 to 2026-07-12

Items per source:
source
gdelt                      241536
hn                         174203
reddit_technology           54530
reddit_wallstreetbets       36711
reddit_stocks               30692
reddit_investing            27414
reddit_SecurityAnalysis      4526
edgar_8k                     3730
stocktwits                   2312


---
## 5. Build Daily Sentiment with Adaptive Rolling Windows

Aggregates to one sentiment score per ticker per day, then computes
adaptive rolling windows where:
- **High momentum or volume surge** → shrink window (use shorter history)
- **Low activity / quiet periods** → expand window (use longer history for stability)

This means the signal is more reactive when something is happening
and more stable when things are quiet.

In [9]:
print('Building daily sentiment aggregation ...')

# Daily aggregate per ticker
daily_raw = (
    df_all_raw.groupby(['ticker','date'])
    .agg(
        story_count    = ('sentiment',     'count'),
        weighted_sent  = ('weighted_sent', 'sum'),
        total_weight   = ('weight',        'sum'),
        max_confidence = ('confidence',    'max'),
        sources_active = ('source',        lambda x: '|'.join(sorted(set(x)))),
        source_count   = ('source',        'nunique'),
        has_8k         = ('source',        lambda x: int('edgar_8k' in x.values)),
        has_stocktwits = ('source',        lambda x: int('stocktwits' in x.values)),
    )
    .reset_index()
)
daily_raw['norm_sentiment'] = (
    daily_raw['weighted_sent'] / daily_raw['total_weight']
).where(daily_raw['total_weight'] > 0)

# EDGAR 8-K boost: multiply sentiment by 1.5 on filing days
daily_raw.loc[daily_raw['has_8k']==1, 'norm_sentiment'] *= 1.5

# Asymmetric weighting: positive x1.5, negative x0.6
daily_raw['norm_sentiment'] = daily_raw['norm_sentiment'].apply(
    lambda s: s * 1.5 if (not pd.isna(s) and s >= 0.05)
              else (s * 0.6 if (not pd.isna(s) and s <= -0.05) else s)
)
daily_raw['date'] = pd.to_datetime(daily_raw['date'])
print(f'Daily aggregate: {len(daily_raw):,} ticker-days')
print(f'Unique tickers : {daily_raw["ticker"].nunique():,}')

Building daily sentiment aggregation ...
Daily aggregate: 164,531 ticker-days
Unique tickers : 2,005


In [10]:
print('Computing adaptive rolling windows ...')

# Build full date grid
all_dates   = pd.date_range(ANALYSIS_START, ANALYSIS_END, freq='D')
all_tickers = daily_raw['ticker'].unique().tolist()
print(f'Building grid: {len(all_tickers):,} tickers x {len(all_dates):,} days')

idx = pd.MultiIndex.from_product([all_tickers, all_dates], names=['ticker','date'])
daily_grid = (
    daily_raw.set_index(['ticker','date'])
    .reindex(idx, fill_value=0)
    .reset_index()
)
daily_grid.loc[daily_grid['story_count']==0, 'norm_sentiment'] = np.nan

signal_frames = []
batch = 100
tickers_list = all_tickers

for i, ticker in enumerate(tickers_list):
    if i % batch == 0:
        print(f'  Processing {i:,}/{len(tickers_list):,} tickers ...', end='\r')

    t = daily_grid[daily_grid['ticker']==ticker].copy().sort_values('date')

    # Volume surge z-score (30-day rolling)
    roll_mean = t['story_count'].rolling(30, min_periods=5).mean()
    roll_std  = t['story_count'].rolling(30, min_periods=5).std()
    t['volume_surge'] = ((t['story_count'] - roll_mean) / (roll_std + 0.01)).fillna(0)

    # Momentum: 3-day rolling vs 7 days ago
    t['roll_3d']      = t['norm_sentiment'].rolling(3, min_periods=1).mean()
    t['sent_momentum']= t['roll_3d'] - t['roll_3d'].shift(7)

    # Adaptive window size per day
    # High volume surge OR high momentum -> smaller window (more reactive)
    # Low activity -> larger window (more stable)
    def adaptive_window(vol_surge, momentum):
        vs = abs(vol_surge) if not np.isnan(vol_surge) else 0
        mo = abs(momentum)  if not np.isnan(momentum)  else 0
        signal_strength = min(1.0, (vs / 3.0) + (mo / 0.3))
        window = int(MAX_WINDOW - signal_strength * (MAX_WINDOW - MIN_WINDOW))
        return max(MIN_WINDOW, min(MAX_WINDOW, window))

    t['adaptive_window'] = [
        adaptive_window(v, m)
        for v, m in zip(t['volume_surge'], t['sent_momentum'].fillna(0))
    ]

    # Compute adaptive rolling sentiment
    sent_values = t['norm_sentiment'].values
    windows     = t['adaptive_window'].values
    adaptive_sent = np.full(len(sent_values), np.nan)
    for j in range(len(sent_values)):
        w   = windows[j]
        seg = sent_values[max(0, j-w+1):j+1]
        valid = seg[~np.isnan(seg)]
        if len(valid) > 0:
            adaptive_sent[j] = valid.mean()
    t['adaptive_sentiment'] = adaptive_sent

    # Fixed windows for comparison
    for w in [1, 3, 5, 7, 21]:
        t[f'roll_{w}d'] = t['norm_sentiment'].rolling(w, min_periods=1).mean()

    signal_frames.append(t)

daily_signals = pd.concat(signal_frames, ignore_index=True)
print(f'\nAdaptive signals complete: {len(daily_signals):,} ticker-days')
print(f'Tickers: {daily_signals["ticker"].nunique():,}')
print(f'Avg adaptive window: {daily_signals["adaptive_window"].mean():.1f} days')
print(f'Signal coverage: {daily_signals["norm_sentiment"].notna().sum():,} days with sentiment')

Computing adaptive rolling windows ...
Building grid: 2,005 tickers x 4,215 days
  Processing 2,000/2,005 tickers ...
Adaptive signals complete: 8,451,075 ticker-days
Tickers: 2,005
Avg adaptive window: 20.3 days
Signal coverage: 162,223 days with sentiment


---
## 6. Save to GitHub

In [11]:
if GITHUB_TOKEN is None:
    print('GITHUB_TOKEN not set')
else:
    print('Saving outputs to GitHub ...')

    # Save daily signals split by quarter to stay under GitHub 100MB limit
    daily_signals['year']    = daily_signals['date'].dt.year
    daily_signals['quarter'] = daily_signals['date'].dt.quarter
    for (year, quarter), df_yq in daily_signals.groupby(['year','quarter']):
        df_out = df_yq.drop(columns=['year','quarter'])
        path   = f'{OUTPUT_PREFIX}/daily_signals_{year}_Q{quarter}.csv'
        push_csv(df_out, path, GITHUB_TOKEN,
                 f'Daily signals {year} Q{quarter}: {len(df_out):,} rows, {df_out["ticker"].nunique()} tickers')

    # Save coverage index
    coverage = (
        daily_signals.groupby('ticker')
        .agg(
            signal_days  = ('norm_sentiment', lambda x: x.notna().sum()),
            date_min     = ('date',           'min'),
            date_max     = ('date',           'max'),
            avg_sentiment= ('norm_sentiment', 'mean'),
            total_stories= ('story_count',    'sum'),
        )
        .reset_index()
        .sort_values('signal_days', ascending=False)
    )
    push_csv(coverage, f'{OUTPUT_PREFIX}/sentiment_coverage.csv', GITHUB_TOKEN,
             f'Sentiment coverage: {len(coverage):,} tickers')

    # Save source attribution summary instead of raw items (too large for GitHub)
    source_summary = (
        df_all_raw.groupby(['ticker','source'])
        .agg(item_count=('sentiment','count'),
             avg_sentiment=('sentiment','mean'),
             avg_confidence=('confidence','mean'))
        .reset_index()
    )
    push_csv(source_summary, f'{OUTPUT_PREFIX}/source_attribution.csv', GITHUB_TOKEN,
             f'Source attribution: {len(source_summary):,} ticker-source pairs')

    print(f'\nAll saved to {OUTPUT_PREFIX}/')
    print(f'  daily_signals_YYYY.csv : one per year')
    print(f'  sentiment_coverage.csv : {len(coverage):,} tickers with signal')
    print(f'  matched_items_raw.csv  : {len(df_all_raw):,} raw matches')
    print(f'\nTop 20 tickers by signal days:')
    print(coverage.head(20)[['ticker','signal_days','total_stories','avg_sentiment']].to_string(index=False))

Saving outputs to GitHub ...
  Saved sentiment_outputs/daily_signals_2015_Q1.csv (180,450 rows)
  Saved sentiment_outputs/daily_signals_2015_Q2.csv (182,455 rows)
  Saved sentiment_outputs/daily_signals_2015_Q3.csv (184,460 rows)
  Saved sentiment_outputs/daily_signals_2015_Q4.csv (184,460 rows)
  Saved sentiment_outputs/daily_signals_2016_Q1.csv (182,455 rows)
  Saved sentiment_outputs/daily_signals_2016_Q2.csv (182,455 rows)
  Saved sentiment_outputs/daily_signals_2016_Q3.csv (184,460 rows)
  Saved sentiment_outputs/daily_signals_2016_Q4.csv (184,460 rows)
  Saved sentiment_outputs/daily_signals_2017_Q1.csv (180,450 rows)
  Saved sentiment_outputs/daily_signals_2017_Q2.csv (182,455 rows)
  Saved sentiment_outputs/daily_signals_2017_Q3.csv (184,460 rows)
  Saved sentiment_outputs/daily_signals_2017_Q4.csv (184,460 rows)
  Saved sentiment_outputs/daily_signals_2018_Q1.csv (180,450 rows)
  Saved sentiment_outputs/daily_signals_2018_Q2.csv (182,455 rows)
  Saved sentiment_outputs/daily_s